# Query Construction (snRNAseq)

In [ ]:
import query_construction_utils_01 as qcu
import pandas as pd
import scanpy as sc
import anndata as ad
import doubletdetection

%matplotlib inline
import matplotlib.pyplot as plt


## 1. Data Input

###  A. Load data and metadata

In [ ]:
# Brain region specific meta data file
meta_data = pd.read_csv('/tscc/nfs/home/aopatel/synapse_meta_NEW/DLPFC_sea_ad_merged_metadata.csv')

# Directory with all cellranger *.h5 files  
base_dir = '/tscc/lustre/ddn/scratch/aopatel/DLPFC_cellranger'


adatas = qcu.read_h5_files(meta_data, base_dir, filtered=True, make_unique=True)



In [ ]:
qcu.check_unique(adatas)

In [ ]:
path = "pre_pre_processing_cellranger_adatas_sizes.csv"
qcu.cell_count_caclulator(adatas, path)

In [ ]:
### 🔵 🚨 DLPFC specific: sample 63, ~36k nuclei, too high (possibly)
### 🔵 🚨 DLPFC specific: sample 86, 329 nuclei, too low (definitely)

# Check which specimen these are
print(f"Sample 63 specimenID: {adatas[63].obs['specimenID'].iloc[0]}")
print(f"Sample 86 specimenID: {adatas[86].obs['specimenID'].iloc[0]}")
print("-----------------------")
# Check metadata for these two suspicious samples
suspicious = ['M1TX_220207_155_H01', 'M2TX_210824_431_A01']

cols_of_interest = ['specimenID', 'individualID', 'comparison_group', 
                    'Severely Affected Donor', 'numberCells', 
                    'medianGenes', 'medianUMIs','totalReads']

print(meta_data[meta_data['specimenID'].isin(suspicious)][cols_of_interest].to_string())

### B. Add metadata to each sample (object in adatas[] list)

<div class="alert alert-block alert-info">
<b> Check sizes of each object in adatas and add metadata correctly!

</div>


In [ ]:
adatas=qcu.meta_data_adder(adatas,meta_data)

In [ ]:
# Quick sanity check on first and last sample
print("=== First sample ===")
print(adatas[0].obs[['specimenID', 'sample_id', 'comparison_group', 'sex', 'RIN', 'CPS']].head(3))
print("\n=== Last sample ===")
print(adatas[99].obs[['specimenID', 'sample_id', 'comparison_group', 'sex', 'RIN', 'CPS']].head(3))

## 2. Quality Control (QC) and Preprocessing

### A. Initial view and assessment

In [ ]:
#### min_genes=1000 
#### to get rid of cells that don't express at least 1000 genes even before any QC 

adatas=[qcu.quality_controller(ad,min_genes=1000, is_indexed=True) for ad in adatas]



In [ ]:
#### Violin plot visualization

for ad in adatas:
    qcu.violin_plots(ad)
    

### B. Preprocessing

In [ ]:
adatas=[qcu.pre_processor(ad,mt_thresh=5, hb_thresh=1) for ad in adatas]

### C. Doublet removal (Scrublet & DoubletDetection)

In [ ]:
#### Create clf object for DoubletDetection

clf = doubletdetection.BoostClassifier(
    n_iters=10,
    clustering_algorithm="leiden",
    standard_scaling=True,
    pseudocount=0.1,
    n_jobs=-1)

In [ ]:
#### Expected doublet rate is estimated from technology used, estimated input and expected yield.
#### Please check associated helper functions page for more info

adatas=[qcu.de_doubletor(ad, to_filter=True, expected_doublet_rate=0.08,
                p_thresh=1e-16, voter_thresh=0.5, clf=clf) for ad in adatas]

In [ ]:
path = "post_pre_processing_cellranger_adatas_sizes.csv"
qcu.cell_count_caclulator(adatas, path)

In [ ]:
# 🔵 🚨 DLPFC specific Remove sample 86 (only 93 cells — unusable)

print(f"Removing sample 86: {adatas[86].obs['specimenID'].iloc[0]} ({adatas[86].n_obs} cells)")
adatas_filtered = [adata for i, adata in enumerate(adatas) if i != 86]
print(f"Samples remaining: {len(adatas_filtered)}")  # should be 99
print(f"Total cells: {sum(ad.n_obs for ad in adatas_filtered)}")

### D. Concatenation

In [ ]:
#### Concatenate all datasets into 1
merged_adata = sc.concat(adatas_filtered, join='outer', index_unique="-") #index is for barcodes that may similar between samples


In [ ]:
merged_adata

In [ ]:
#### sc.concat removes .vars data 
#### Is is critical to attach .vars data CORRECTLY!

## grab all var DataFrames from our list
all_var = [x.var for x in adatas_filtered]
## concatenate them
all_var = pd.concat(all_var, join="outer")
## remove duplicates
all_var = all_var[~all_var.index.duplicated()]

## assign
merged_adata.var = all_var.loc[merged_adata.var_names]

In [ ]:
print(merged_adata)

### E. Gene Filter and re-calculate QC metrics (its okay if the %s are different, we are removing some genes)

In [ ]:
#### Add gene filter
sc.pp.filter_genes(merged_adata, min_cells=25)

#### Recalculate qc metrics now that file is merged
sc.pp.calculate_qc_metrics(merged_adata, qc_vars=["mt", "ribo", "hb"], inplace=True, percent_top=[20], log1p=True)

In [ ]:
## Another sanity check!
merged_adata.var

<div class="alert alert-block alert-info">
<b> Save this file. After reference is constructed, we will proceed with this file (as query) for cell type classification next

</div>

In [ ]:
# Save progress up to this point, just in case
merged_adata.write_h5ad("/tscc/lustre/ddn/scratch/aopatel/preprocessed_adata_DLPFC.h5ad")